# Movie Recommendation System Analysis

This notebook provides detailed analysis and experimentation with the recommendation system.

In [ ]:
# Import required libraries
import sys
sys.path.append('..')

from recommendation_system import MovieRecommendationSystem
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Initialize and Load Data

In [ ]:
# Initialize the recommendation system
rec_system = MovieRecommendationSystem()
rec_system.load_data()
rec_system.prepare_data()
rec_system.calculate_similarities()

## 2. Data Exploration

In [ ]:
# Basic statistics
print("Dataset Overview:")
print(f"Number of users: {rec_system.ratings_df['userId'].nunique()}")
print(f"Number of movies: {rec_system.ratings_df['movieId'].nunique()}")
print(f"Number of ratings: {len(rec_system.ratings_df)}")
print(f"Rating range: {rec_system.ratings_df['rating'].min()} - {rec_system.ratings_df['rating'].max()}")
print(f"Average rating: {rec_system.ratings_df['rating'].mean():.2f}")

In [ ]:
# Rating distribution
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
rec_system.ratings_df['rating'].value_counts().sort_index().plot(kind='bar')
plt.title('Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Count')

plt.subplot(2, 2, 2)
user_rating_counts = rec_system.ratings_df['userId'].value_counts()
plt.hist(user_rating_counts, bins=50, alpha=0.7)
plt.title('Ratings per User Distribution')
plt.xlabel('Number of Ratings')
plt.ylabel('Number of Users')

plt.subplot(2, 2, 3)
movie_rating_counts = rec_system.ratings_df['movieId'].value_counts()
plt.hist(movie_rating_counts, bins=50, alpha=0.7)
plt.title('Ratings per Movie Distribution')
plt.xlabel('Number of Ratings')
plt.ylabel('Number of Movies')

plt.subplot(2, 2, 4)
avg_ratings = rec_system.ratings_df.groupby('movieId')['rating'].mean()
plt.hist(avg_ratings, bins=30, alpha=0.7)
plt.title('Average Movie Ratings Distribution')
plt.xlabel('Average Rating')
plt.ylabel('Number of Movies')

plt.tight_layout()
plt.show()

## 3. Model Training and Evaluation

In [ ]:
# Train SVD model
rmse, mae = rec_system.train_svd_model()
print(f"SVD Model Performance:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

## 4. Recommendation Analysis

In [ ]:
# Compare different recommendation methods
sample_users = list(rec_system.user_encoder.keys())[:5]
methods = ['svd', 'user_based', 'item_based']

for user_id in sample_users:
    print(f"\n=== USER {user_id} RECOMMENDATIONS ===")
    
    for method in methods:
        recs = rec_system.get_recommendations(user_id, method, 3)
        print(f"\n{method.upper()} Method:")
        for i, (movie_id, rating) in enumerate(recs, 1):
            movie_title = rec_system.movies_df[rec_system.movies_df['movieId'] == movie_id]['title'].iloc[0]
            print(f"  {i}. {movie_title} ({rating:.2f})")

## 5. Similarity Analysis

In [ ]:
# Analyze similarity distributions
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
# Sample user similarities (avoid plotting full matrix)
sample_user_similarities = rec_system.user_similarity[:100, :100]
plt.hist(sample_user_similarities.flatten(), bins=50, alpha=0.7)
plt.title('User Similarity Distribution (Sample)')
plt.xlabel('Cosine Similarity')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
# Sample item similarities
sample_item_similarities = rec_system.item_similarity[:100, :100]
plt.hist(sample_item_similarities.flatten(), bins=50, alpha=0.7)
plt.title('Item Similarity Distribution (Sample)')
plt.xlabel('Cosine Similarity')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

## 6. Performance Comparison

In [ ]:
# Evaluate all models
results = rec_system.evaluate_models()

if results:
    # Create comparison plot
    models = list(results.keys())
    rmse_scores = [results[model]['RMSE'] for model in models]
    mae_scores = [results[model]['MAE'] for model in models]
    
    plt.figure(figsize=(10, 5))
    
    plt.subplot(1, 2, 1)
    plt.bar(models, rmse_scores, alpha=0.7)
    plt.title('Model Comparison - RMSE')
    plt.ylabel('RMSE')
    plt.xticks(rotation=45)
    
    plt.subplot(1, 2, 2)
    plt.bar(models, mae_scores, alpha=0.7, color='orange')
    plt.title('Model Comparison - MAE')
    plt.ylabel('MAE')
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Print results
    for model, metrics in results.items():
        print(f"{model}: RMSE={metrics['RMSE']:.4f}, MAE={metrics['MAE']:.4f}")

## 7. Cold Start Analysis

In [ ]:
# Analyze cold start problem
popular_movies = rec_system.analyze_cold_start()

# Show popular movies for new users
print("Popular Movies for Cold Start Users:")
for i, movie_id in enumerate(popular_movies[:10], 1):
    movie_title = rec_system.movies_df[rec_system.movies_df['movieId'] == movie_id]['title'].iloc[0]
    movie_genre = rec_system.movies_df[rec_system.movies_df['movieId'] == movie_id]['genres'].iloc[0]
    print(f"{i}. {movie_title} ({movie_genre})")

## 8. Conclusion

This analysis demonstrates the effectiveness of different recommendation approaches:

1. **SVD (Matrix Factorization)**: Provides good overall performance and handles sparse data well
2. **User-based Collaborative Filtering**: Works well for users with similar preferences
3. **Item-based Collaborative Filtering**: Good for discovering similar items

The system successfully addresses the cold start problem by recommending popular movies to new users.